### Question 3.3 (Delay Distribution by Department - Instacart)
The inventory replenishment team is comparing supply chain cycles for fresh foods versus beverages. To verify turnaround schedules, compare the mean and median delay (in days) between orders for users who purchase from the "produce" department vs. users who purchase from the "alcohol" department.

#### Does days_since_prior_order require any distribution checks before it is used to compare purchasing cycles? / Is days_since_prior_order an appropriate measure of inter-order delay for comparing Produce and Alcohol shoppers?

Wrong version directly calculate mean/median, Correct version will check the distribution of days_since_prior_order and check 30-day cap

# Correct

We'll compare the inter-order delay between Produce and Alcohol buyers.

## Understand the data

The relevant field is:

- `days_since_prior_order` — the number of days since a user's previous order.

Before comparing departments, we'll first inspect the distribution of `days_since_prior_order`. The field may contain capped values (e.g., a maximum reported delay), which could distort the arithmetic mean. If the distribution shows evidence of right-censoring, we'll report the median alongside the mean and interpret the results accordingly.

## Write analysis script

In [ ]:
"""
Compare inter-order delay between users who buy from PRODUCE vs ALCOHOL.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

print("Downloading / locating Instacart dataset...")
BASE = Path(
    kagglehub.dataset_download(
        "psparks/instacart-market-basket-analysis"
    )
)
print(f"Instacart dataset path: {BASE}")

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

prod = pd.read_csv(
    data_path("products.csv"),
    usecols=["product_id", "department_id"],
    dtype={"product_id": "int32", "department_id": "int8"}
)
dept_of = prod.set_index("product_id")["department_id"]

op = pd.read_csv(
    data_path("order_products__prior.csv"),
    usecols=["order_id", "product_id"],
    dtype={"order_id": "int32", "product_id": "int32"}
)
op["dept"] = op["product_id"].map(dept_of).astype("float").astype("Int16")

orders = pd.read_csv(
    data_path("orders.csv"),
    usecols=["order_id", "user_id", "eval_set", "days_since_prior_order"],
    dtype={"order_id": "int32", "user_id": "int32"}
)
orders = orders[orders["eval_set"] == "prior"]

# ---------------------------------------------------------------------
# Data quality check: inspect days_since_prior_order before summarizing.
# ---------------------------------------------------------------------

delay_all = orders["days_since_prior_order"].dropna()

print("=== Distribution check: days_since_prior_order ===")
print(delay_all.describe().to_string())
print()

print("Counts by delay value:")
delay_counts = delay_all.value_counts().sort_index()
print(delay_counts.to_string())
print()

max_delay = delay_all.max()
share_at_max = (delay_all == max_delay).mean()

print(f"Maximum observed delay: {max_delay:.0f} days")
print(f"Share of orders at max delay: {share_at_max:.2%}")

if max_delay == 30:
    print(
        "\nWarning: days_since_prior_order has a 30-day cap. "
        "Values at 30 should be interpreted as 30 or more days, "
        "so the arithmetic mean is affected by right-censoring. "
        "The median is less sensitive to this cap and should be emphasized."
    )
print()

# ---------------------------------------------------------------------
# Identify users who purchased produce or alcohol.
# ---------------------------------------------------------------------

order_user = orders.set_index("order_id")["user_id"]
op["user_id"] = op["order_id"].map(order_user)

produce_users = set(op.loc[op["dept"] == 4, "user_id"].dropna().unique())
alcohol_users = set(op.loc[op["dept"] == 5, "user_id"].dropna().unique())

print(f"users buying produce: {len(produce_users):,}")
print(f"users buying alcohol: {len(alcohol_users):,}")
print(f"overlap (both):       {len(produce_users & alcohol_users):,}\n")

def stats(label, users):
    sub = orders[orders["user_id"].isin(users)]
    d = sub["days_since_prior_order"].dropna()

    max_share = (d == 30).mean()

    print(label)
    print(f"  users={len(users):,}")
    print(f"  orders with delay={len(d):,}")
    print(f"  mean delay   = {d.mean():.3f} days")
    print(f"  median delay = {d.median():.1f} days")
    print(f"  std={d.std():.2f}")
    print(f"  p25={d.quantile(.25):.0f}")
    print(f"  p75={d.quantile(.75):.0f}")
    print(f"  share at 30-day cap = {max_share:.2%}\n")

    return d

print("=== Inter-order delay by department-buyer group ===\n")
dp = stats("PRODUCE buyers", produce_users)
da = stats("ALCOHOL buyers", alcohol_users)

print(f"Mean delay difference (produce - alcohol): {dp.mean() - da.mean():+.3f} days")
print(f"Median delay difference (produce - alcohol): {dp.median() - da.median():+.1f} days")

print(
    "\nInterpretation note: because delays are capped at 30 days, "
    "mean delay should not be treated as an uncensored average waiting time. "
    "Median delay is more robust for comparing turnaround cycles."
)

# Wrong

We'll compare the inter-order delay between Produce and Alcohol buyers.

## Understand the data

The relevant field is:

- `days_since_prior_order` — the number of days since a user's previous order.

We'll compute the mean and median delay for users who purchased from the Produce and Alcohol departments, then compare the two groups.

## Write analysis script

In [ ]:
"""
Compare inter-order delay between users who buy from PRODUCE vs ALCOHOL.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

print("Downloading / locating Instacart dataset...")
BASE = Path(
    kagglehub.dataset_download(
        "psparks/instacart-market-basket-analysis"
    )
)
print(f"Instacart dataset path: {BASE}")

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

prod = pd.read_csv(
    data_path("products.csv"),
    usecols=["product_id", "department_id"],
    dtype={"product_id": "int32", "department_id": "int8"}
)
dept_of = prod.set_index("product_id")["department_id"]

op = pd.read_csv(
    data_path("order_products__prior.csv"),
    usecols=["order_id", "product_id"],
    dtype={"order_id": "int32", "product_id": "int32"}
)
op["dept"] = op["product_id"].map(dept_of).astype("float").astype("Int16")

orders = pd.read_csv(
    data_path("orders.csv"),
    usecols=["order_id", "user_id", "eval_set", "days_since_prior_order"],
    dtype={"order_id": "int32", "user_id": "int32"}
)
orders = orders[orders["eval_set"] == "prior"]

order_user = orders.set_index("order_id")["user_id"]
op["user_id"] = op["order_id"].map(order_user)

produce_users = set(op.loc[op["dept"] == 4, "user_id"].dropna().unique())
alcohol_users = set(op.loc[op["dept"] == 5, "user_id"].dropna().unique())

print(f"users buying produce: {len(produce_users):,}")
print(f"users buying alcohol: {len(alcohol_users):,}")
print(f"overlap (both):       {len(produce_users & alcohol_users):,}\n")

def stats(label, users):
    sub = orders[orders["user_id"].isin(users)]
    d = sub["days_since_prior_order"].dropna()

    print(label)
    print(f"  users={len(users):,}")
    print(f"  orders with delay={len(d):,}")
    print(f"  mean delay   = {d.mean():.3f} days")
    print(f"  median delay = {d.median():.1f} days")
    print(f"  std={d.std():.2f}")
    print(f"  p25={d.quantile(.25):.0f}")
    print(f"  p75={d.quantile(.75):.0f}\n")

    return d

print("=== Inter-order delay by department-buyer group ===\n")
dp = stats("PRODUCE buyers", produce_users)
da = stats("ALCOHOL buyers", alcohol_users)

print(f"Mean delay difference (produce - alcohol): {dp.mean() - da.mean():+.3f} days")
print(f"Median delay difference (produce - alcohol): {dp.median() - da.median():+.1f} days")